# CSET485 - AI and Society: Assignment #1
**Student Name:** Vansh Jain | **Enrollment No:** S24CSEU0161  
**Assigned Dataset:** COMPAS Recidivism Risk Score Dataset  
**Random Seed:** 161 (70% Train, 30% Test)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('figures', exist_ok=True)
os.makedirs('data', exist_ok=True)

## Data Preprocessing and Cleaning

In [ ]:
df = pd.read_csv('data/compas-scores-two-years.csv')

df_clean = df[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != 'O') &
    (df['score_text'] != 'N/A')
].copy()

df_clean['sex_male'] = (df_clean['sex'] == 'Male').astype(int)
df_clean['charge_felony'] = (df_clean['c_charge_degree'] == 'F').astype(int)
df_clean['race_black'] = (df_clean['race'] == 'African-American').astype(int)
df_clean['race_caucasian'] = (df_clean['race'] == 'Caucasian').astype(int)
df_clean['two_year_recid'] = df_clean['two_year_recid'].astype(int)

features = ['age', 'priors_count', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'sex_male', 'charge_felony', 'race_black']
target_A = 'decile_score'
target_B = 'two_year_recid'

train_df, test_df = train_test_split(df_clean, test_size=0.3, random_state=161)
print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

## Part 1 — Build and Model Comparison

In [ ]:
X_train = train_df[features]
y_train_A = train_df[target_A]
X_test = test_df[features]
y_test_A = test_df[target_A]

lr = LinearRegression()
lr.fit(X_train, y_train_A)
preds_A = lr.predict(X_test)
med_A = float(np.median(preds_A))
class_A = (preds_A >= med_A).astype(int)

y_train_B = train_df[target_B]
y_test_B = test_df[target_B]

logr = LogisticRegression(max_iter=1000, random_state=161)
logr.fit(X_train, y_train_B)
probs_B = logr.predict_proba(X_test)[:, 1]
class_B = (probs_B >= 0.5).astype(int)

cm = confusion_matrix(class_A, class_B)
mismatches = int(np.sum(class_A != class_B))
total_test = len(class_A)
pct_mismatch = float((mismatches / total_test) * 100)

print(f'Model A median threshold: {med_A:.4f}')
print('Confusion Matrix (Row=Model A [0=Low, 1=High], Col=Model B [0=Low, 1=High]):')
print(cm)
print(f'Mismatched count: {mismatches} / {total_test} ({pct_mismatch:.2f}%)')

## Part 2.1 — Simpson's Paradox Hunt

In [ ]:
def fit_logit(data, feats):
    X = sm.add_constant(data[feats])
    y = data['two_year_recid']
    return sm.Logit(y, X).fit(disp=False)

age_feat = ['priors_count', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'sex_male', 'charge_felony', 'race_black']
sub_age_A = df_clean[df_clean['age'] < 25]
sub_age_B = df_clean[df_clean['age'] >= 25]

full_age_res = fit_logit(df_clean, age_feat)
sub_age_A_res = fit_logit(sub_age_A, age_feat)
sub_age_B_res = fit_logit(sub_age_B, age_feat)

records_simpson = []
for f in age_feat:
    records_simpson.append({
        'Feature': f,
        'Full Coeff': round(float(full_age_res.params[f]), 4),
        'Full p-val': f'{full_age_res.pvalues[f]:.4e}',
        'Age<25 Coeff': round(float(sub_age_A_res.params[f]), 4),
        'Age<25 p-val': f'{sub_age_A_res.pvalues[f]:.4e}',
        'Age>=25 Coeff': round(float(sub_age_B_res.params[f]), 4),
        'Age>=25 p-val': f'{sub_age_B_res.pvalues[f]:.4e}'
    })
simpson_table = pd.DataFrame(records_simpson)
simpson_table

## Part 2.2 — Omitted-Variable Bias

In [ ]:
full_model_feats = ['age', 'sex_male', 'charge_felony', 'race_black', 'priors_count']
red_model_feats = ['age', 'sex_male', 'charge_felony', 'race_black']

full_logit = fit_logit(train_df, full_model_feats)
red_logit = fit_logit(train_df, red_model_feats)

ovb_table = []
for f in red_model_feats:
    c_full = full_logit.params[f]
    c_red = red_logit.params[f]
    pct_change = ((c_red - c_full) / abs(c_full)) * 100
    ovb_table.append({
        'Feature': f,
        'Full Model Coeff': round(c_full, 4),
        'Reduced Model Coeff': round(c_red, 4),
        'Shift': round(c_red - c_full, 4),
        'Pct Change (%)': round(pct_change, 2)
    })

ovb_df = pd.DataFrame(ovb_table)
ovb_df

## Part 2.3 — Adversarial Subset Construction

In [ ]:
test_results = test_df.copy()
test_results['prob'] = probs_B
test_results['pred_class'] = class_B
test_results['actual'] = y_test_B

adv_fp = test_results[(test_results['prob'] >= 0.75) & (test_results['actual'] == 0)]
adv_fn = test_results[(test_results['prob'] <= 0.25) & (test_results['actual'] == 1)]
adv_all = pd.concat([adv_fp, adv_fn])

total_adv = len(adv_all)
pct_adv = (total_adv / len(test_results)) * 100
print(f'High-confidence wrong predictions: {total_adv} / {len(test_results)} ({pct_adv:.2f}%)')
print(f'False Positives (p>=0.75, actual=0): {len(adv_fp)}')
print(f'False Negatives (p<=0.25, actual=1): {len(adv_fn)}')

adv_50 = adv_all.sample(n=50, random_state=161)
adv_50[['age', 'sex', 'race', 'priors_count', 'prob', 'actual']].head(10)

## Part 2.4 — Fairness Trade-Off Analysis

In [ ]:
test_fair = test_results[test_results['race'].isin(['African-American', 'Caucasian'])].copy()
group_A = test_fair[test_fair['race'] == 'African-American']
group_B = test_fair[test_fair['race'] == 'Caucasian']

thresholds = [np.round(x, 2) for x in np.arange(0.05, 1.05, 0.1) if x < 1.0]

fairness_records = []
for t in thresholds:
    pred_A = (group_A['prob'] >= t).astype(int)
    pred_B = (group_B['prob'] >= t).astype(int)
    
    dp_num = pred_A.mean()
    dp_den = pred_B.mean()
    dp = dp_num / dp_den if dp_den > 0 else np.nan
    
    tpr_A = ((pred_A == 1) & (group_A['actual'] == 1)).sum() / (group_A['actual'] == 1).sum() if (group_A['actual'] == 1).sum() > 0 else 0
    tpr_B = ((pred_B == 1) & (group_B['actual'] == 1)).sum() / (group_B['actual'] == 1).sum() if (group_B['actual'] == 1).sum() > 0 else 0
    
    fpr_A = ((pred_A == 1) & (group_A['actual'] == 0)).sum() / (group_A['actual'] == 0).sum() if (group_A['actual'] == 0).sum() > 0 else 0
    fpr_B = ((pred_B == 1) & (group_B['actual'] == 0)).sum() / (group_B['actual'] == 0).sum() if (group_B['actual'] == 0).sum() > 0 else 0
    
    ratio_tpr = tpr_A / tpr_B if tpr_B > 0 else 0
    ratio_fpr = fpr_A / fpr_B if fpr_B > 0 else 0
    eo = min(ratio_tpr, ratio_fpr)
    
    prec_A = ((pred_A == 1) & (group_A['actual'] == 1)).sum() / pred_A.sum() if pred_A.sum() > 0 else np.nan
    prec_B = ((pred_B == 1) & (group_B['actual'] == 1)).sum() / pred_B.sum() if pred_B.sum() > 0 else np.nan
    pp = prec_A / prec_B if (not np.isnan(prec_A) and not np.isnan(prec_B) and prec_B > 0) else np.nan
    
    pred_all = (test_fair['prob'] >= t).astype(int)
    acc = accuracy_score(test_fair['actual'], pred_all)
    
    fairness_records.append({
        'threshold': t,
        'DP': round(dp, 4),
        'EO': round(eo, 4),
        'PP': round(pp, 4) if not np.isnan(pp) else np.nan,
        'Accuracy': round(acc, 4),
        'TPR_A': round(tpr_A, 4),
        'TPR_B': round(tpr_B, 4),
        'FPR_A': round(fpr_A, 4),
        'FPR_B': round(fpr_B, 4)
    })

fair_df = pd.DataFrame(fairness_records)

plt.figure(figsize=(9, 6))
valid_pts = fair_df.dropna(subset=['DP', 'EO'])
plt.scatter(valid_pts['DP'], valid_pts['EO'], color='darkblue', s=70, zorder=5)
for _, row in valid_pts.iterrows():
    plt.annotate(f"t={row['threshold']:.2f}", (row['DP'], row['EO']),
                 textcoords="offset points", xytext=(8, 4), fontsize=9)
plt.axvline(1.0, color='red', linestyle='--', alpha=0.7, label='Ideal DP = 1.0')
plt.axhline(1.0, color='green', linestyle='--', alpha=0.7, label='Ideal EO = 1.0')
plt.xlim(0, 2.5)
plt.ylim(0, 1.5)
plt.xlabel('Demographic Parity (DP)')
plt.ylabel('Equalized Odds (EO)')
plt.title('Fairness Trade-Off: DP vs EO across Decision Thresholds (COMPAS)')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('figures/fairness_tradeoff.png', dpi=300)
plt.show()

fair_df[['threshold', 'DP', 'EO', 'PP', 'Accuracy', 'TPR_A', 'TPR_B', 'FPR_A', 'FPR_B']]